###Ejercicio 1):
 Utilice lo aprendido en los trabajos prácticos previos (uso de biopython para obtenersecuencias, BLAST y el “parser del mismo”) para:i) obtener 10 secuencias pertenecientes a una misma familia yii) Alinearlas y obtener la correspondiente matriz de puntaje / identidadAnalice si la matriz es (o no) simetricaCompare matrices utilizando diferentes puntajes que le otorga BLAST (Bit Score, Identidad, E-Value).*texto en cursiva*

Lista de 10 proteinas CYP1A1 de distintas especies

In [35]:
!pip install biopython

In [69]:
from Bio import SeqIO, pairwise2
from Bio.Align import substitution_matrices
import numpy as np
import pandas as pd


In [41]:
!wget ftp://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ncbi-blast-*-x64-linux.tar.gz
!tar -xzf ncbi-blast-*-x64-linux.tar.gz
!mv ncbi-blast-*/bin/* /usr/local/bin/


--2025-09-18 13:24:45--  ftp://ftp.ncbi.nlm.nih.gov/blast/executables/blast+/LATEST/ncbi-blast-*-x64-linux.tar.gz
           => ‘.listing’
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.11, 130.14.250.12, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /blast/executables/blast+/LATEST ... done.
==> PASV ... done.    ==> LIST ... done.

.listing                [ <=>                ]   2.62K  --.-KB/s    in 0.02s   

2025-09-18 13:24:46 (144 KB/s) - ‘.listing’ saved [2678]

Removed ‘.listing’.
--2025-09-18 13:24:46--  ftp://ftp.ncbi.nlm.nih.gov/blast/executables/blast%2B/LATEST/ncbi-blast-2.17.0%2B-x64-linux.tar.gz
           => ‘ncbi-blast-2.17.0+-x64-linux.tar.gz’
==> CWD not required.
==> PASV ... done.    ==> RETR ncbi-blast-2.17.0+-x64-linux.tar.gz ... done.
Length: 296006458 (282M)

ncbi-blas

In [57]:


import pandas as pd

def cargar_blosum60(path="/content/blosum60.txt"):
    with open(path) as f:
        lines = [line.strip() for line in f if not line.startswith("#")]

    header = lines[0].split()
    data = []
    index = []

    for line in lines[1:]:
        parts = line.split()
        index.append(parts[0])
        data.append([int(x) for x in parts[1:]])

    df = pd.DataFrame(data, index=index, columns=header)
    return df

# Cargar matriz
blosum60_df = cargar_blosum60()

# Mostrar parte de la matriz

def df_to_blosum_dict(df):
    matrix = {}
    for i in df.index:
        for j in df.columns:
            score = df.at[i, j]
            matrix[(i, j)] = score
            matrix[(j, i)] = score  # aseguramos simetría
    return matrix


blosum_dict = df_to_blosum_dict(blosum60_df)
print(blosum_dict)


{('A', 'A'): np.int64(5), ('A', 'R'): np.int64(-2), ('R', 'A'): np.int64(-2), ('A', 'N'): np.int64(-1), ('N', 'A'): np.int64(-1), ('A', 'D'): np.int64(-2), ('D', 'A'): np.int64(-2), ('A', 'C'): np.int64(-1), ('C', 'A'): np.int64(-1), ('A', 'Q'): np.int64(-1), ('Q', 'A'): np.int64(-1), ('A', 'E'): np.int64(-1), ('E', 'A'): np.int64(-1), ('A', 'G'): np.int64(0), ('G', 'A'): np.int64(0), ('A', 'H'): np.int64(-2), ('H', 'A'): np.int64(-2), ('A', 'I'): np.int64(-2), ('I', 'A'): np.int64(-2), ('A', 'L'): np.int64(-2), ('L', 'A'): np.int64(-2), ('A', 'K'): np.int64(-1), ('K', 'A'): np.int64(-1), ('A', 'M'): np.int64(-1), ('M', 'A'): np.int64(-1), ('A', 'F'): np.int64(-3), ('F', 'A'): np.int64(-3), ('A', 'P'): np.int64(-1), ('P', 'A'): np.int64(-1), ('A', 'S'): np.int64(1), ('S', 'A'): np.int64(1), ('A', 'T'): np.int64(0), ('T', 'A'): np.int64(0), ('A', 'W'): np.int64(-4), ('W', 'A'): np.int64(-4), ('A', 'Y'): np.int64(-2), ('Y', 'A'): np.int64(-2), ('A', 'V'): np.int64(0), ('V', 'A'): np.int6

In [71]:

def cargar_blosum60(path="/content/blosum60.txt"):
    with open(path) as f:
        lines = [line.strip() for line in f if not line.startswith("#")]

    header = lines[0].split()
    data = []
    index = []

    for line in lines[1:]:
        parts = line.split()
        index.append(parts[0])
        data.append([int(x) for x in parts[1:]])

    df = pd.DataFrame(data, index=index, columns=header)
    return df

# Cargar matriz
blosum60_df = cargar_blosum60()

files = [f"g{i}.fasta" for  i in range(1,10)]

# Leer las secuencias
sequences = {}
for file in files:
    for record in SeqIO.parse(file, "fasta"):
        sequences[file] = str(record.seq)
        break

# Parámetros de alineamiento
gap_open = -10
gap_extend = -0.5

# Inicializar matriz
n = len(files)
matrix = np.zeros((n, n))

# Alineamientos globales entre todos los pares
for i in range(n):
    for j in range(n):
        seq1 = sequences[files[i]]
        seq2 = sequences[files[j]]
       # if i == j:
        #    matrix[i, j] = 1.0  # Identidad consigo mismo
        #else:
        alignment = pairwise2.align.globalds(seq1, seq2, blosum_dict, gap_open, gap_extend, one_alignment_only=True)
        score = alignment[0].score if alignment else 0
        matrix[i, j] = score

# Mostrar matriz
labels = [f.split(".")[0] for f in files]
df_matriz_pares = pd.DataFrame(matrix, index=labels, columns=labels)
print(df_matriz_pares)
print()
print(matrix)
display(matrix)

        g1      g2      g3      g4      g5      g6      g7      g8      g9
g1  1496.0    15.0   626.0     3.5   -43.5    30.0    22.0   -29.0   -29.5
g2    15.0  1490.0    79.5    35.0   -22.5   383.0    12.5    10.0    66.0
g3   626.0    79.5  1351.0    34.0    14.5    69.0    76.0    12.0     1.5
g4     3.5    35.0    34.0  1523.0     9.5    53.5     5.0    56.0    38.0
g5   -43.5   -22.5    14.5     9.5  1465.0   -11.5   -20.5    -4.5   -14.0
g6    30.0   383.0    69.0    53.5   -11.5  1507.0    62.0    45.5     9.5
g7    22.0    12.5    76.0     5.0   -20.5    62.0  1493.0    35.5    36.5
g8   -29.0    10.0    12.0    56.0    -4.5    45.5    35.5  1446.0   253.5
g9   -29.5    66.0     1.5    38.0   -14.0     9.5    36.5   253.5  1540.0

[[ 1.496e+03  1.500e+01  6.260e+02  3.500e+00 -4.350e+01  3.000e+01
   2.200e+01 -2.900e+01 -2.950e+01]
 [ 1.500e+01  1.490e+03  7.950e+01  3.500e+01 -2.250e+01  3.830e+02
   1.250e+01  1.000e+01  6.600e+01]
 [ 6.260e+02  7.950e+01  1.351e+03  3.400

array([[ 1.496e+03,  1.500e+01,  6.260e+02,  3.500e+00, -4.350e+01,
         3.000e+01,  2.200e+01, -2.900e+01, -2.950e+01],
       [ 1.500e+01,  1.490e+03,  7.950e+01,  3.500e+01, -2.250e+01,
         3.830e+02,  1.250e+01,  1.000e+01,  6.600e+01],
       [ 6.260e+02,  7.950e+01,  1.351e+03,  3.400e+01,  1.450e+01,
         6.900e+01,  7.600e+01,  1.200e+01,  1.500e+00],
       [ 3.500e+00,  3.500e+01,  3.400e+01,  1.523e+03,  9.500e+00,
         5.350e+01,  5.000e+00,  5.600e+01,  3.800e+01],
       [-4.350e+01, -2.250e+01,  1.450e+01,  9.500e+00,  1.465e+03,
        -1.150e+01, -2.050e+01, -4.500e+00, -1.400e+01],
       [ 3.000e+01,  3.830e+02,  6.900e+01,  5.350e+01, -1.150e+01,
         1.507e+03,  6.200e+01,  4.550e+01,  9.500e+00],
       [ 2.200e+01,  1.250e+01,  7.600e+01,  5.000e+00, -2.050e+01,
         6.200e+01,  1.493e+03,  3.550e+01,  3.650e+01],
       [-2.900e+01,  1.000e+01,  1.200e+01,  5.600e+01, -4.500e+00,
         4.550e+01,  3.550e+01,  1.446e+03,  2.535e+02],
